# MAHA-GR-ALIGN — MuRIL conflict-classifier fine-tune

**How to run (T4 GPU):** Runtime → Change runtime type → T4 GPU. Upload
`pairs_train.jsonl`, `pairs_dev.jsonl`, `pairs_test.jsonl` to `/content/`
(drag into the Files pane), then **Runtime → Run all** (~20 min).
Outputs: `metrics_finetuned.json`, `confusion_matrix.png`, `muril_gr_conflict_fp16.zip`
(auto-copied to Google Drive if you allow the mount cell).

In [ ]:
!pip install -q "transformers==4.57.*" datasets accelerate scikit-learn matplotlib

In [ ]:
import json, numpy as np, torch, random
from pathlib import Path
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
LABEL2ID = {"contradiction": 0, "consistent": 1, "neutral": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
def load(split):
    rows = [json.loads(l) for l in Path(f"/content/pairs_{split}.jsonl").read_text(encoding="utf-8").splitlines()]
    return rows
train_rows, dev_rows, test_rows = load("train"), load("dev"), load("test")
print({s: len(r) for s, r in [("train", train_rows), ("dev", dev_rows), ("test", test_rows)]})

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
MODEL = "google/muril-base-cased"
tok = AutoTokenizer.from_pretrained(MODEL)
def to_ds(rows):
    ds = Dataset.from_dict({
        "premise": [r["premise"] for r in rows],
        "hypothesis": [r["hypothesis"] for r in rows],
        "label": [LABEL2ID[r["label"]] for r in rows]})
    return ds.map(lambda b: tok(b["premise"], b["hypothesis"], truncation=True, max_length=256),
                  batched=True, remove_columns=["premise", "hypothesis"])
dtrain, ddev, dtest = to_ds(train_rows), to_ds(dev_rows), to_ds(test_rows)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback
from sklearn.metrics import f1_score, accuracy_score
def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro"),
            "accuracy": accuracy_score(p.label_ids, preds)}
args = TrainingArguments(
    output_dir="/content/out", seed=SEED,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    learning_rate=2e-5, num_train_epochs=5, warmup_ratio=0.1, weight_decay=0.01,
    fp16=True, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    save_total_limit=1, logging_steps=25, report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=dtrain, eval_dataset=ddev,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
pred = trainer.predict(dtest)
yh = pred.predictions.argmax(-1); y = pred.label_ids
rep = classification_report(y, yh, target_names=list(LABEL2ID), output_dict=True, zero_division=0)
import transformers, sklearn
metrics = {"macro_f1": f1_score(y, yh, average="macro"), "accuracy": accuracy_score(y, yh),
           "report": rep, "seed": SEED, "model": MODEL,
           "versions": {"transformers": transformers.__version__, "sklearn": sklearn.__version__}}
Path("/content/metrics_finetuned.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps({k: metrics[k] for k in ("macro_f1", "accuracy")}, indent=2))
cm = confusion_matrix(y, yh)
ConfusionMatrixDisplay(cm, display_labels=list(LABEL2ID)).plot(cmap="Blues")
plt.title("MuRIL fine-tuned — GR clause conflict (test)")
plt.savefig("/content/confusion_matrix.png", dpi=150, bbox_inches="tight")

In [ ]:
# Export fp16 weights + tokenizer as one zip
import shutil
model.half().save_pretrained("/content/muril_gr_conflict_fp16")
tok.save_pretrained("/content/muril_gr_conflict_fp16")
shutil.copy("/content/metrics_finetuned.json", "/content/muril_gr_conflict_fp16/")
shutil.make_archive("/content/muril_gr_conflict_fp16", "zip", "/content/muril_gr_conflict_fp16")
print("zip size MB:", round(Path("/content/muril_gr_conflict_fp16.zip").stat().st_size/1e6))

In [ ]:
# Optional: copy artifacts to Google Drive (approve the mount prompt)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/maha-gr-align/"
    import os; os.makedirs(dest, exist_ok=True)
    for f in ["muril_gr_conflict_fp16.zip", "metrics_finetuned.json", "confusion_matrix.png"]:
        shutil.copy(f"/content/{f}", dest)
    print("copied to Drive:", dest)
except Exception as e:
    print("Drive copy skipped:", e)